In [40]:
import pickle
import numpy as np
import os
import polars as pl
import json
from yagm.utils.geometric import batch_perspective_transform_2d
from PIL import Image
from ecg.utils.misc import get_scale_xy_from_homo_mat

In [ ]:
with open('/home/dangnh36/projects/ecg/outputs/pseudo_label_raw_round1_finetune_dsnt/fold0_predictions.pkl', 'rb') as f:
    data = pickle.load(f)
data.keys()

dict_keys(['gt_df', 'main_kpt_preds', 'grid_kpt_pred'])

In [7]:
MAIN_KPT_METHOD = 'dsnt'

gt_df = data['gt_df']
all_main_kpt_pred = data['main_kpt_preds'][MAIN_KPT_METHOD]
all_grid_kpt_pred = data['grid_kpt_pred']
print(all_main_kpt_pred.shape, len(all_grid_kpt_pred))
gt_df

(1764, 57, 2) 1764


index,id,type_id,rot_code,H
u32,i64,i64,f64,str
9,540379822,1,null,null
10,540379822,3,null,"""[[1.0467598347304883, -0.01341…"
11,540379822,4,null,"""[[1.0453547953194353, -0.00579…"
12,540379822,5,null,"""[[0.6804241554108705, 0.067108…"
13,540379822,6,null,"""[[0.6681521080978683, 0.038194…"
…,…,…,…,…
8698,3726383111,6,null,"""[[0.788007335600758, -0.003682…"
8699,3726383111,9,null,"""[[0.6852612431859918, 0.005565…"
8700,3726383111,10,null,"""[[0.6045926860705563, 0.002720…"


In [8]:
with open('/home/dangnh36/datasets/ecg/processed/reference_keypoints.json', 'r') as f:
    ref_kpts = json.load(f)
ref_kpt_xys = np.array(list(ref_kpts.values()))
ref_kpt_names = list(ref_kpts.keys())
len(ref_kpt_names), ref_kpt_xys.shape

(2422, (2422, 2))

In [9]:
ref_grid_kpt_xys = ref_kpt_xys[:2365]
ref_grid_min_x = ref_grid_kpt_xys[:, 0].min()
ref_grid_max_x = ref_grid_kpt_xys[:, 0].max()
ref_grid_min_y = ref_grid_kpt_xys[:, 1].min()
ref_grid_max_y = ref_grid_kpt_xys[:, 1].max()


ref_safe_corners = np.array([
    [ref_grid_min_x - 20, ref_grid_min_y - 20],
    [ref_grid_max_x + 20, ref_grid_min_y - 20],
    [ref_grid_max_x + 20, ref_grid_max_y + 20],
    [ref_grid_min_x - 20, ref_grid_max_y + 20]
], dtype = np.float32)
ref_safe_corners

array([[  19.370079,  -12.913385],
       [2185.3542  ,  -12.913385],
       [2185.3542  , 1680.6299  ],
       [  19.370079, 1680.6299  ]], dtype=float32)

In [10]:
import cv2
import numpy as np


def get_mask_opencv(keypoints, polygon_coords):
    """
    Generates a boolean mask for keypoints lying inside a polygon using OpenCV.

    Args:
        keypoints (np.ndarray): Shape (L, 2) in (x, y) order. Can be float or int.
        polygon_coords (np.ndarray): Shape (4, 2) in (x, y) order.
                                     Defined as [Top-Left, Top-Right, Bottom-Right, Bottom-Left].

    Returns:
        np.ndarray: A boolean mask of shape (L,) where True indicates the point 
                    is inside or on the edge of the polygon.
    """
    # 1. OpenCV contours require int32 (for pixel precision) or float32.
    # We cast to int32 here as per your provided snippet, which is standard for image polygons.
    poly_array = np.array(polygon_coords, dtype=np.int32)
    
    mask = []
    
    # 2. Iterate through the (L, 2) array
    for point in keypoints:
        # pointPolygonTest requires the point as a tuple (x, y).
        # We allow float coordinates for the points even if the polygon is int.
        pt_tuple = (float(point[0]), float(point[1]))
        
        # measureDist=False returns:
        # +1 if inside, -1 if outside, 0 if on the edge.
        result = cv2.pointPolygonTest(poly_array, pt_tuple, measureDist=False)
        
        # We consider "inside" or "on edge" (>= 0) as True
        mask.append(result >= 0)
        
    return np.array(mask, dtype=bool)

In [11]:
import numpy as np
from numba import njit

@njit
def match_single_image_full(pred_xy, pred_score, gt_xy, threshold):
    """
    Greedy confidence-ordered matching for ONE image.
    
    Returns:
        matches (np.ndarray): Shape (K, 2) [Pred Index, GT Index]
        unmatched_preds (np.ndarray): Shape (P-K, ) Indices of unmatched predictions
        unmatched_gts (np.ndarray): Shape (G-K, ) Indices of unmatched ground truths
    """
    P = pred_xy.shape[0]
    G = gt_xy.shape[0]

    # 1. Sort predictions by score (descending)
    order = np.argsort(-pred_score)
    sorted_pred_xy = pred_xy[order]
    
    # 2. Initialize tracking
    gt_used = np.zeros(G, dtype=np.bool_)
    pred_used_sorted_idx = np.zeros(P, dtype=np.bool_) # Tracks usage in sorted order
    
    # Pre-allocate matches array (max matches = min(P, G))
    max_matches = min(P, G)
    temp_matches = np.zeros((max_matches, 2), dtype=np.int64)
    match_count = 0

    # 3. Greedy Matching Loop
    for i in range(P):
        if G == 0:
            break

        min_dist = 1e12
        min_j = -1

        # Find closest unused GT
        for j in range(G):
            if gt_used[j]:
                continue
                
            dx = gt_xy[j, 0] - sorted_pred_xy[i, 0]
            dy = gt_xy[j, 1] - sorted_pred_xy[i, 1]
            d = (dx * dx + dy * dy) ** 0.5
            
            if d < min_dist:
                min_dist = d
                min_j = j

        # Check threshold
        if min_j >= 0 and min_dist <= threshold:
            gt_used[min_j] = True
            pred_used_sorted_idx[i] = True
            
            # Store match: (Original Pred Index, GT Index)
            temp_matches[match_count, 0] = order[i]
            temp_matches[match_count, 1] = min_j
            match_count += 1

    # 4. Finalize Matches
    matches = temp_matches[:match_count]

    # 5. Determine Unmatched Indices
    # Unmatched GTs: straightforward, just check the boolean mask
    # Numba supports boolean indexing like NumPy
    all_gt_indices = np.arange(G)
    unmatched_gts = all_gt_indices[~gt_used]

    # Unmatched Preds: need to map back to original indices
    # We kept track of 'pred_used' in the *sorted* order.
    # We need to find which indices in 'order' were NOT marked as used.
    
    # Create a boolean mask for original indices
    pred_used_original = np.zeros(P, dtype=np.bool_)
    for i in range(P):
        if pred_used_sorted_idx[i]:
            original_idx = order[i]
            pred_used_original[original_idx] = True
            
    all_pred_indices = np.arange(P)
    unmatched_preds = all_pred_indices[~pred_used_original]
    
    return matches, unmatched_preds, unmatched_gts

In [12]:
import numpy as np
import scipy.optimize
import scipy.spatial
from typing import Tuple

# Small epsilon to handle floating point comparisons or gating logic
EPSILON = 1e-5

def linear_assignment_hungarian(
        cost_matrix: np.ndarray,
        max_cost: float) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Linear assignment using Scipy's Hungarian method implementation.
    Provided implementation adapted for the wrapper.
    """
    # Create a copy to avoid modifying the original cost matrix input
    cost_matrix = cost_matrix.copy()

    # Gate costs: treated every over-range cost equally to remove bias
    cost_matrix[cost_matrix > max_cost] = max_cost + EPSILON

    # Perform the Hungarian Algorithm (Munkres algorithm)
    # Returns (row_indices, col_indices)
    row_ind, col_ind = scipy.optimize.linear_sum_assignment(cost_matrix)
    
    matches = []
    for ia, ib in zip(row_ind, col_ind):
        # Filter out matches that exceed the maximum allowed threshold
        if cost_matrix[ia, ib] <= max_cost:
            matches.append([ia, ib])
    
    matches = np.asarray(matches)

    # Determine unmatched indices
    if len(matches) > 0:
        unmatched_a = np.asarray([
            idx for idx in range(cost_matrix.shape[0])
            if idx not in matches[:, 0]
        ])
        unmatched_b = np.asarray([
            idx for idx in range(cost_matrix.shape[1])
            if idx not in matches[:, 1]
        ])
    else:
        # If no matches, everything is unmatched
        unmatched_a = np.arange(cost_matrix.shape[0])
        unmatched_b = np.arange(cost_matrix.shape[1])
        # Ensure matches is an empty array of shape (0, 2) for consistency
        matches = np.empty((0, 2), dtype=int)

    return matches, unmatched_a, unmatched_b

def match_single_image_hungarian(pred_xy, pred_score, gt_xy, threshold):
    """
    Optimal (Hungarian) matching for ONE image.
    
    Args:
        pred_xy (np.ndarray): Shape (P, 2)
        pred_score (np.ndarray): Shape (P,) - Not used for matching logic in Hungarian, 
                                 kept for interface consistency.
        gt_xy (np.ndarray): Shape (G, 2)
        threshold (float): Maximum euclidean distance allowed for a match.

    Returns:
        matches (np.ndarray): Shape (K, 2) [Pred Index, GT Index]
        unmatched_preds (np.ndarray): Shape (P-K, ) Indices of unmatched predictions
        unmatched_gts (np.ndarray): Shape (G-K, ) Indices of unmatched ground truths
    """
    P = pred_xy.shape[0]
    G = gt_xy.shape[0]

    # Handle Edge Cases: Empty inputs
    if P == 0:
        return np.empty((0, 2), dtype=int), np.empty(0, dtype=int), np.arange(G)
    if G == 0:
        return np.empty((0, 2), dtype=int), np.arange(P), np.empty(0, dtype=int)

    # 1. Compute Cost Matrix (Euclidean Distance)
    # Shape will be (P, G)
    cost_matrix = scipy.spatial.distance.cdist(pred_xy, gt_xy, metric='euclidean')

    # 2. Perform Hungarian Matching
    matches, unmatched_preds, unmatched_gts = linear_assignment_hungarian(cost_matrix, threshold)

    return matches, unmatched_preds, unmatched_gts

In [13]:
import cv2
import numpy as np

def visualize_matches(image_rgb, matches, unmatched_preds, unmatched_gts, pred_xy, gt_xy):
    """
    Visualizes matching results on the provided image.
    
    Args:
        image_rgb (np.ndarray): Input image (H, W, 3).
        matches (np.ndarray): Shape (K, 2) [Pred Index, GT Index].
        unmatched_preds (np.ndarray): Indices of unmatched predictions.
        unmatched_gts (np.ndarray): Indices of unmatched ground truths.
        pred_xy (np.ndarray): All prediction coordinates (P, 2).
        gt_xy (np.ndarray): All ground truth coordinates (G, 2).
        
    Returns:
        np.ndarray: The annotated image.
    """
    # 1. Setup Canvas
    # OpenCV uses BGR by default, so if input is RGB, convert it or just treat channels accordingly.
    # We will assume we want to return a BGR image for cv2.imwrite/imshow usage.
    # If image_rgb is None, create a black canvas 1700x2300 as requested.
    if image_rgb is None:
        canvas = np.zeros((1700, 2300, 3), dtype=np.uint8)
    else:
        # Convert RGB to BGR for OpenCV drawing functions
        canvas = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    # 2. Define Colors (BGR format)
    COLOR_MATCH = (0, 255, 0)      # Green
    COLOR_UNMATCH_PRED = (0, 0, 255) # Red (False Positive)
    COLOR_UNMATCH_GT = (255, 0, 0)   # Blue (False Negative / Missed)
    
    # 3. Draw Unmatched Ground Truths (Blue Circles)
    if gt_xy is not None:
        for idx in unmatched_gts:
            pt = tuple(gt_xy[idx].astype(int))
            # Draw a hollow circle to represent GT
            cv2.circle(canvas, pt, radius=8, color=COLOR_UNMATCH_GT, thickness=2)

    # 4. Draw Unmatched Predictions (Red Crosses/Dots)
    for idx in unmatched_preds:
        pt = tuple(pred_xy[idx].astype(int))
        # Draw a filled circle to represent Prediction
        cv2.circle(canvas, pt, radius=5, color=COLOR_UNMATCH_PRED, thickness=-1)

    # 5. Draw Matches (Green Lines connecting Pred to GT)
    for (pred_idx, gt_idx) in matches:
        pt_pred = tuple(pred_xy[pred_idx].astype(int))
        if gt_xy is not None:
            pt_gt = tuple(gt_xy[gt_idx].astype(int))
            # Draw Line
            cv2.line(canvas, pt_pred, pt_gt, COLOR_MATCH, thickness=2)
            # GT as larger hollow circle
            cv2.circle(canvas, pt_gt, radius=8, color=COLOR_MATCH, thickness=2)
        
        # Draw endpoints to see the shift
        # Pred as small filled dot
        cv2.circle(canvas, pt_pred, radius=4, color=COLOR_MATCH, thickness=-1)
        

    return canvas

In [14]:
43 * 55, ref_grid_kpt_xys.shape

(2365, (2365, 2))

In [15]:
### LOAD RGB IMAGE
sample = gt_df[0].to_dicts()[0]
sample_id = sample["id"]
type_id = sample["type_id"]
rot_code = sample["rot_code"]

img_path = os.path.join(
    '/home/dangnh36/datasets/ecg/raw/train/', str(sample_id), f"{sample_id}-{type_id:04d}.png"
)
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
# correct the orientation
if rot_code is not None:
    assert rot_code == int(rot_code)
    rot_code = int(rot_code)
    img = cv2.rotate(img, rot_code)

REF_IMG = img

In [72]:
from tqdm import tqdm


CONF_THRES = 0.1


for i in tqdm(range(len(gt_df))):
    sample = gt_df[i].to_dicts()[0]
    grid_kpt = all_grid_kpt_pred[i]
    if len(grid_kpt) != 2365:
        # print('original', i, grid_kpt.shape, grid_kpt[:, 2].min(), grid_kpt[:, 2].mean(), grid_kpt[:, 2].max())
        grid_kpt = grid_kpt[grid_kpt[:, 2] > CONF_THRES]
    if len(grid_kpt) != 2365:
        # print('after thres', i, grid_kpt.shape, grid_kpt[:, 2].min(), grid_kpt[:, 2].mean(), grid_kpt[:, 2].max())
        pass
        
    # map back to ref coordinates
    if sample['H'] is None:
        ref_grid_kpt_pred = grid_kpt.copy()[:, :2]
        from_ref_H = None
    else:
        to_ref_H = np.array(eval(sample["H"]))
        from_ref_H = np.linalg.pinv(to_ref_H)
        ref_grid_kpt_pred = batch_perspective_transform_2d(grid_kpt[None, :, :2], to_ref_H[None])[0]
        # print(ref_grid_kpt_pred.shape)
        
    # filter outside keypoints
    mask = get_mask_opencv(ref_grid_kpt_pred, ref_safe_corners)
    # print(mask.sum())
    ref_grid_kpt_pred = ref_grid_kpt_pred[mask]
    grid_kpt = grid_kpt[mask]

    if len(grid_kpt) == 2365:
        continue
    
    # print('after outside', i, grid_kpt.shape, grid_kpt[:, 2].min(), grid_kpt[:, 2].mean(), grid_kpt[:, 2].max())
    
    match_func = match_single_image_hungarian
    # match_func = match_single_image_full

    THRES1 = 15
    matches, unmatched_preds, unmatched_gts = match_func(ref_grid_kpt_pred, grid_kpt[:, 2], ref_grid_kpt_xys, threshold = THRES1)
    # print('match1', len(matches), len(unmatched_preds), len(unmatched_gts))

    # sample_id = sample["id"]
    # type_id = sample["type_id"]
    # rot_code = sample["rot_code"]
    
    # img_path = os.path.join(
    #     '/home/dangnh36/datasets/ecg/raw/train/', str(sample_id), f"{sample_id}-{type_id:04d}.png"
    # )
    # img = cv2.imread(img_path)
    # img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # # correct the orientation
    # if rot_code is not None:
    #     assert rot_code == int(rot_code)
    #     rot_code = int(rot_code)
    #     img = cv2.rotate(img, rot_code)

    # viz = visualize_matches(img, matches, unmatched_preds, unmatched_gts, grid_kpt[:, :2], None)
    # display(Image.fromarray(viz))
    
    # viz = visualize_matches(REF_IMG, matches, unmatched_preds, unmatched_gts, ref_grid_kpt_pred, ref_grid_kpt_xys)
    # display(Image.fromarray(viz))

    # simply ignore unmatched predictions, then
    # ========== HANDLE UNMATCHED GT =========

    # format: pred_idx, gt_idx, pred_x, pred_y, gt_x, gt_y
    matches = np.concatenate([matches,
                              grid_kpt[matches[:, 0], :2],
                              ref_grid_kpt_xys[matches[:, 1]]
                             ], axis = 1)
    # format: x, y, index
    if len(unmatched_gts) > 0:
        unmatched_gts = np.concatenate([ref_grid_kpt_xys[unmatched_gts], unmatched_gts[:, None]], axis = 1)
    else:
        unmatched_gts = []
    if len(unmatched_preds) > 0:
        # format: x, y, index
        unmatched_ref_preds = np.concatenate([ref_grid_kpt_pred[unmatched_preds], unmatched_preds[:, None]], axis=1)
        # format: x, y, conf, index
        unmatched_preds = np.concatenate([grid_kpt[unmatched_preds], unmatched_preds[:, None]], axis=1)
    else:
        unmatched_ref_preds = np.empty((0, 3), dtype = int)
        unmatched_preds = np.empty((0, 3), dtype = int)

    interpolated_xys = []
    for ref_gt_point in unmatched_gts:
        # find  top 24 nearest GT points
        cur_gt_xy = ref_gt_point[:2]
        cur_gt_idx = ref_gt_point[2]
        dists = cur_gt_xy[None] - matches[:, 4:6]
        dists = (dists[:, 0] ** 2 + dists[:, 1] ** 2) ** 0.5
        sort_idxs = np.argsort(dists)
        nearest_idxs = sort_idxs[:24]
        nearest_matches = matches[nearest_idxs]
        # find local matching homo from REF TO CURRENT
        src_pts = nearest_matches[:, 4:6]
        dst_pts = nearest_matches[:, 2:4]
    
        # 2. Find Homography
        # USAC_MAGSAC: Uses the MAGSAC++ algorithm
        # ransacReprojThreshold: Even though MAGSAC is "threshold-free", OpenCV still requires 
        # this parameter as an upper bound for internal optimizations. 3.0 - 5.0 is standard.
        H, mask = cv2.findHomography(
            src_pts, 
            dst_pts, 
            cv2.USAC_MAGSAC, 
            ransacReprojThreshold=8.0, 
            maxIters=5000, 
            confidence=0.9999
        )
        if mask.sum() < 24:
            # print('WARNING: local homo matrix estimation resulted in unmatched pairs', mask.sum(), '/', mask.size)
            pass
            
        interpolated_pred_xy = cv2.perspectiveTransform(cur_gt_xy.reshape(1,1,2), H)[0,0]
        # print(interpolated_pred_xy)
        interpolated_xys.append(interpolated_pred_xy)
    interpolated_xys = np.array(interpolated_xys)

    # roughtly estimate the scale from current original image (000x) relative to the reference image 0001
    THRES2 = 8
    if from_ref_H is not None:
        ori_scale_x, ori_scale_y = get_scale_xy_from_homo_mat(from_ref_H)
        thres2 = THRES2 * ((ori_scale_x ** 2 + ori_scale_y ** 2) ** 0.5)
    else:
        thres2 = THRES2
    matches2, unmatched_preds2, unmatched_gts2 = match_func(
        unmatched_preds[:, :2],
        unmatched_preds[:, 2],
        interpolated_xys, threshold = thres2)
    # print('match2', len(matches2), len(unmatched_preds2), len(unmatched_gts2))

    final = np.zeros_like(ref_grid_kpt_xys)
    final[matches[:, 1].astype(np.uint32)] = matches[:, 2:4]
    matched2_gt_idxs = unmatched_gts[:, 2][matches2[:, 1]].astype(np.uint32)
    if len(unmatched_gts2) > 0:
        unmatched2_gt_idxs = unmatched_gts[:, 2][unmatched_gts2].astype(np.uint32)
    else:
        unmatched2_gt_idxs = np.empty((0,), dtype = np.uint32)
    final[matched2_gt_idxs] = unmatched_preds[:, :2][matches2[:, 0].astype(np.uint32)]
    if len(unmatched_gts2) > 0:
        final[unmatched2_gt_idxs] = interpolated_xys[unmatched_gts2]
    assert len(matches) + len(matched2_gt_idxs) + len(unmatched2_gt_idxs) == 2365
    assert not np.any(np.all(final == 0.0, axis=1), axis = 0)
    
    
    
    
    
    
    
        
        



        
        
    
    
    
    # break
    

    # if i >= 100:
    #     break

100%|█| 1764/1764 [00:46<00:00, 37.99it/
